# GrainLens
## Physical texture to grain-size curves

**Small data. Measurable texture. Valid distributions.**

GrainLens converts soil photographs into a physical-scale texture fingerprint and predicts a cumulative grain-size curve with a lightweight model. No pretrained backbone is needed: five Difference-of-Gaussians energies, a square-root mass representation and Ridge regression form a compact, inspectable pipeline.

This notebook contributes an end-to-end CPU baseline, camera-aware image scaling, valid CDF construction, and evaluations that keep photographs of the same soil together. The camera-transfer tests make the generalization gap visible rather than hiding it in a random photo split.

[Source and companion files](https://github.com/GISWLH/AImodel-EarthScience/tree/main/kaggle-geoscience/grainlens) · Author: **giswlh / Longhao Wang**, with AI assistance.


In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.dpi':115,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold'})
OUTPUT=Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()/'outputs'
OUTPUT.mkdir(parents=True,exist_ok=True)


## The recipe

1. Use the camera's pixels-per-millimetre specification and the actual image dimensions to align physical scale.
2. Measure normalized DoG energy at six Gaussian scales: 0.25, 0.5, 1, 2, 4, 8 mm. Their five adjacent differences describe texture at different particle sizes.
3. Convert cumulative mass labels into bin masses. Learn their square roots with sample-balanced Ridge; square, normalize and accumulate predictions to produce a valid CDF.
4. Fit all statistical preprocessing inside the training fold. Group every photograph of the held soil together.

The fixed choices here are 4.5 pixels/mm, central-half crops and Ridge alpha 1. They are a transparent reference configuration, not a parameter search.


In [ ]:
"""Train-only, standalone physical texture baseline and grouped domain audit.
Original educational implementation by giswlh, AI-assisted, 2026-10-19.
"""
import os
from pathlib import Path
import re
import unicodedata
import json
import platform
import hashlib
import time
import numpy as np
import pandas as pd
import scipy
from scipy.ndimage import gaussian_filter
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import sklearn
from PIL import Image, ImageOps
import PIL

START=time.monotonic()
DIAMETERS=np.array([.002,.0063,.02,.063,.2,.63,2,6.3,20,63,200])
COLUMNS=['0.002','0.0063','0.02','0.063','0.2','0.63','2','6.3','20','63','200']
TARGET_PPM=4.5
ALPHA=1.0

def canonical(text):
    text=str(text).translate(str.maketrans({'ü':'ue','ö':'oe','ä':'ae','ß':'ss'}))
    return re.sub('[^a-z0-9]','',unicodedata.normalize('NFKD',text).encode('ascii','ignore').decode().lower())

def emd(truth,prediction):
    return np.sum(np.abs(np.asarray(truth)[:,:-1]-np.asarray(prediction)[:,:-1])*np.diff(np.log10(DIAMETERS)),axis=1)

def check_cdf(values):
    a=np.asarray(values)
    assert a.ndim==2 and a.shape[1]==11 and np.isfinite(a).all()
    assert a.min()>=0 and a.max()<=100 and (np.diff(a,axis=1)>=-1e-10).all()
    assert np.all(a[:,-1]==100.)

def extract_features(root,labels):
    ppm=pd.read_csv(root/'ppm_updated.csv').set_index('phone')
    ids=list(labels.index);phones=sorted(ppm.index,key=lambda p:len(canonical(p)),reverse=True);records=[]
    # Only Training directories are opened. No test photographs or hidden labels.
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in {'.jpg','.jpeg'}:continue
        if not any('training' in part.lower() for part in path.relative_to(root).parts):continue
        key=canonical(path.stem);matches=[sid for sid in ids if canonical(sid) in key]
        phone=next((p for p in phones if key.startswith(canonical(p))),None)
        if len(matches)!=1 or phone is None:raise ValueError('Ambiguous training photograph mapping: '+path.name)
        with Image.open(path) as im:
            # Scale from the original file dimensions before EXIF orientation.
            effective_ppm=float(ppm.loc[phone,'ppm'])*max(im.size)/max(ppm.loc[phone,'width'],ppm.loc[phone,'height'])
            image=ImageOps.exif_transpose(im).convert('L')
            scale=TARGET_PPM/effective_ppm
            image=image.resize((max(8,round(image.width*scale)),max(8,round(image.height*scale))),Image.Resampling.LANCZOS)
            gray=np.asarray(image,dtype=np.float32)
        height,width=gray.shape;gray=gray[height//4:3*height//4,width//4:3*width//4]
        contrast=float(gray.std())+1e-6
        smooth=[gaussian_filter(gray,sigma=mm*TARGET_PPM) for mm in [.25,.5,1,2,4,8]]
        features=[float(np.log(np.std(a-b)/contrast+1e-6)) for a,b in zip(smooth[:-1],smooth[1:])]
        records.append({'sample_id':matches[0],'phone':phone,'file':path.name,**{f'dog_{i}':v for i,v in enumerate(features)}})
    frame=pd.DataFrame(records)
    assert set(frame.sample_id)==set(ids),'Missing training photo coverage'
    assert np.isfinite(frame.filter(like='dog_').to_numpy()).all()
    return frame

def proxy_sites(ids):
    # Development grouping heuristic only: these are NOT verified geographic sites.
    group=0;previous=None;result={}
    for sid in sorted(ids,key=lambda s:(s[0],int(s[1:]))):
        if previous is None or sid[0]!=previous[0] or int(sid[1:])-int(previous[1:])>12:group+=1
        result[sid]=group;previous=sid
    return result

def predict(fit,valid,labels):
    assert not set(fit.sample_id)&set(valid.sample_id),'Soil sample leakage'
    cols=[f'dog_{i}' for i in range(5)]
    # All fitted statistics and model weights are fold-local; each soil has total weight1.
    weights=1/fit.groupby('sample_id').sample_id.transform('size').to_numpy()
    scaler=StandardScaler().fit(fit[cols],sample_weight=weights)
    x=scaler.transform(fit[cols]);v=scaler.transform(valid[cols])
    y=labels.loc[fit.sample_id].to_numpy()/100
    mass=np.diff(np.c_[np.zeros(len(y)),y],axis=1)
    assert mass.min()>=-1e-10
    model=Ridge(alpha=ALPHA).fit(x,np.sqrt(np.maximum(mass,0)),sample_weight=weights)
    root_mass=np.maximum(model.predict(v),0);mass=root_mass**2
    zero=mass.sum(axis=1)<=1e-12;mass[zero]=1.
    mass/=mass.sum(axis=1,keepdims=True)
    curve=np.cumsum(mass,axis=1)*100;curve=np.clip(curve,0,100);curve[:,-1]=100
    result=pd.DataFrame(curve,index=valid.sample_id,columns=COLUMNS).groupby(level=0).mean()
    check_cdf(result);return result

def evaluate(photos,labels,sites):
    protocols={
        'leave_one_soil_out':{'groups':{s:s for s in labels.index},'train_phone':None,'valid_phone':None},
        'leave_proxy_site_out':{'groups':sites,'train_phone':None,'valid_phone':None},
        'Motorola_to_Samsung':{'groups':sites,'train_phone':'Motorola Edge','valid_phone':'Samsung A52'},
        'Samsung_to_Motorola':{'groups':sites,'train_phone':'Samsung A52','valid_phone':'Motorola Edge'},
    }
    summaries=[];oof_rows=[]
    for name,cfg in protocols.items():
        available=photos if cfg['valid_phone'] is None else photos[photos.phone==cfg['valid_phone']]
        predicted=[]
        for group in sorted(set(cfg['groups'].values()),key=str):
            held={s for s,g in cfg['groups'].items() if g==group}
            fit=photos[~photos.sample_id.isin(held)];valid=available[available.sample_id.isin(held)]
            if cfg['train_phone'] is not None:fit=fit[fit.phone==cfg['train_phone']]
            if valid.empty:continue
            assert len(set(fit.sample_id))>=2 and not set(fit.sample_id)&held
            p=predict(fit,valid,labels);error=emd(labels.loc[p.index].to_numpy(),p.to_numpy())
            for sid,e,curve in zip(p.index,error,p.to_numpy()):
                predicted.append(float(e));oof_rows.append({'protocol':name,'sample_id':sid,'held_group':group,'emd':float(e),**dict(zip(COLUMNS,curve.tolist()))})
        assert len(predicted)==available.sample_id.nunique()
        summaries.append({'protocol':name,'n_soils':len(predicted),'mean_emd':float(np.mean(predicted)),'worst_soil_emd':float(max(predicted))})
    return summaries,pd.DataFrame(oof_rows)

def run_soil_training_audit():
    override=os.environ.get('SOIL_DATA_ROOT')
    if override:root=Path(override)
    else:
        files=list(Path('/kaggle/input').rglob('Training_labels_updated.csv'))
        if len(files)!=1:raise RuntimeError('Attach soil-grain-size-from-photos competition data; expected one updated training label file.')
        root=files[0].parent
    out=Path(os.environ.get('SOIL_OUTPUT_ROOT','/kaggle/working'));out.mkdir(parents=True,exist_ok=True)
    labels=pd.read_csv(root/'Training_labels_updated.csv',dtype={'sample_id':str}).set_index('sample_id')
    labels.columns=[format(float(c),'g') for c in labels.columns]
    labels=labels[COLUMNS];check_cdf(labels)
    photos=extract_features(root,labels);sites=proxy_sites(labels.index)
    summary,oof=evaluate(photos,labels,sites)
    report={'method':'fixed physical DoG + sqrt-mass Ridge','hyperparameters':{'target_ppm':TARGET_PPM,'alpha':ALPHA,'gaussian_scales_mm':[.25,.5,1,2,4,8]},'n_training_soils':len(labels),'n_training_photos':len(photos),'n_proxy_groups':len(set(sites.values())),'results':summary,'formal_submission':False,'leaderboard_score':None,'test_photos_read':False,'labels_sha256':hashlib.file_digest((root/'Training_labels_updated.csv').open('rb'),'sha256').hexdigest(),'environment':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__,'Pillow':PIL.__version__},'elapsed_seconds':time.monotonic()-START}
    (out/'validation_report.json').write_text(json.dumps(report,indent=2,allow_nan=False)+'\n');oof.to_csv(out/'grouped_oof.csv',index=False)
    print(pd.DataFrame(summary).to_string(index=False));print(json.dumps({k:report[k] for k in ['n_training_soils','n_training_photos','n_proxy_groups','elapsed_seconds','test_photos_read','formal_submission']},indent=2))
    return report,oof



## Run the real-data camera audit

The code reads labelled training photographs only. “Proxy groups” join nearby same-letter sample IDs with adjacent numeric gaps of at most 12; they are a development heuristic, not verified geography. The two transfer directions train on one known camera and evaluate the other camera on different soils. This is a known-device stress test, not a blind new-phone trial.


In [ ]:
if not os.environ.get('SOIL_DATA_ROOT'):
    matches=list(Path('/kaggle/input').rglob('Training_labels_updated.csv'))
    if len(matches)!=1:raise RuntimeError('Attach the Soil Grain Size competition data.')
    os.environ['SOIL_DATA_ROOT']=str(matches[0].parent)
os.environ['SOIL_OUTPUT_ROOT']=str(OUTPUT/'grainlens')
report,oof=run_soil_training_audit()
summary=pd.DataFrame(report['results'])
display(summary)


## See the camera gap

EMD is the area between cumulative curves on a log-diameter axis. Lower is better. The four protocols answer different questions and their sample coverage is shown explicitly; read them together.


In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
colors=['#20a39e','#2274a5','#ffbf69','#ef8354']
ax.barh(summary.protocol,summary.mean_emd,color=colors)
ax.set_xlabel('Log-diameter-weighted EMD (lower is better)');ax.set_title('One compact model, four generalization tests')
for i,row in summary.iterrows():ax.text(row.mean_emd+.3,i,f'{row.mean_emd:.2f} | {row.n_soils} soils',va='center')
ax.set_xlim(0,summary.mean_emd.max()*1.3);plt.tight_layout();plt.show()


## Inspect actual held-soil curves

These predictions are out of fold: each plotted soil and all of its photographs were excluded from fitting. The examples below are selected by sorted ID, not by their error.


In [ ]:
labels=pd.read_csv(Path(os.environ['SOIL_DATA_ROOT'])/'Training_labels_updated.csv').set_index('sample_id')
labels.columns=[format(float(c),'g') for c in labels.columns]
examples=sorted(labels.index)[:3];pred=oof[oof.protocol=='leave_proxy_site_out'].set_index('sample_id')
fig,axes=plt.subplots(1,3,figsize=(13,3.6),sharey=True)
for ax,sid in zip(axes,examples):
    ax.semilogx(DIAMETERS,labels.loc[sid,COLUMNS],'-o',color='#2274a5',label='Lab measurement')
    ax.semilogx(DIAMETERS,pred.loc[sid,COLUMNS].to_numpy(dtype=float),'--s',color='#ef8354',label='Held-soil prediction')
    ax.set(title=sid,xlabel='Grain diameter (mm)',ylim=(0,103));ax.grid(alpha=.2)
axes[0].set_ylabel('Cumulative mass (%)');axes[-1].legend(fontsize=8);plt.tight_layout();plt.show()


## What to build on

The strongest contribution of a small-data baseline is a reliable comparison. Keep soil grouping and camera tests fixed when adding segmentation, learned embeddings or optical calibration. Inspect failures at the coarse and fine ends separately: visible surface area and sieve mass are different measurements.

**Credits:** [YogaNukko's physical texture and SAM study](https://www.kaggle.com/code/takahitomizunobyts/soil-texture-model-sam-cobble-measurement) and [Nomannic's sqrt-mass Ridge approach](https://www.kaggle.com/code/nomannic19/cv-psgsdi-dinov2-features-sqrt-mass-ridge) inspired the recipe. This is an independent lightweight implementation; no third-party neural weights are bundled. Competition data retain their own access and use conditions.
